In [1]:
import os
import gc
import time
import numpy as np
import pandas as pd
import tensorflow as tf
from datetime import datetime

from train import process_channel
from utils.visualizer import create_summary_visualization


In [2]:
def main():
    """Main execution function with cross-attention hierarchical multi-scale approach"""
    
    # Set random seeds for reproducibility
    tf.random.set_seed(42)
    np.random.seed(42)
    
    # Define parameters
    use_feature_engineering = True
    feature_dim = 20
    
    # Define paths to data - use path joining for cross-platform compatibility
    parent_dir = os.path.dirname(os.path.dirname(os.getcwd()))
    channels_folder = os.path.join(parent_dir, 'data', 'raw', 'train')
    labeled_anomalies_file = os.path.join(parent_dir, 'data', 'processed', 'final_predictions.csv')
    
    # Create output directory with timestamp
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_dir = f"memory_bank_results_{timestamp}"
    os.makedirs(output_dir, exist_ok=True)

    # Create plots subdirectory for visualizations
    plots_dir = os.path.join(output_dir, "plots")
    os.makedirs(plots_dir, exist_ok=True)
    
    # Load labeled anomalies
    labeled_anomalies = pd.read_csv(labeled_anomalies_file)
    channels = labeled_anomalies['chan_id'].unique()
    print(f"Found {len(channels)} channels in the labeled anomalies file")
    
    # Create empty DataFrame for storing results
    results_df = pd.DataFrame(
        columns=[
            'channel', 
            'f1_with_memory', 'precision_with_memory', 'recall_with_memory',
            'tp_with_memory', 'fp_with_memory', 'tn_with_memory', 'fn_with_memory',
            'f1_without_memory', 'precision_without_memory', 'recall_without_memory',
            'tp_without_memory', 'fp_without_memory', 'tn_without_memory', 'fn_without_memory',
            'improvement', 'memory_usage', 'scales', 'processing_time'
        ]
    )
    
    # Process channels
    print(f"Processing {len(channels)} channels with memory bank integration...")
    overall_start_time = time.time()
        
    for i, stream in enumerate(channels):
        print(f"\n[{i+1}/{len(channels)}] Processing {stream}...")
        channel_start_time = time.time()
        
        try:
            # Process channel with memory bank
            metrics = process_channel(
                stream, 
                feature_dimensionality=feature_dim,
                use_feature_engineering=use_feature_engineering,
                plots_dir=plots_dir  # Pass the plots directory
            )
            
            # Calculate processing time
            processing_time = time.time() - channel_start_time
            
            if metrics is not None:
                # Add to results DataFrame with all metrics
                results_df.loc[len(results_df)] = {
                    'channel': stream,
                    'f1_with_memory': metrics['f1_with_memory'],
                    'precision_with_memory': metrics['precision_with_memory'],
                    'recall_with_memory': metrics['recall_with_memory'],
                    'tp_with_memory': metrics['tp_with_memory'],
                    'fp_with_memory': metrics['fp_with_memory'],
                    'tn_with_memory': metrics['tn_with_memory'],
                    'fn_with_memory': metrics['fn_with_memory'],
                    'f1_without_memory': metrics['f1_without_memory'],
                    'precision_without_memory': metrics['precision_without_memory'],
                    'recall_without_memory': metrics['recall_without_memory'],
                    'tp_without_memory': metrics['tp_without_memory'],
                    'fp_without_memory': metrics['fp_without_memory'],
                    'tn_without_memory': metrics['tn_without_memory'],
                    'fn_without_memory': metrics['fn_without_memory'],
                    'improvement': metrics['improvement'],
                    'memory_usage': str(metrics['memory_usage']),
                    'scales': str(metrics['scales']),
                    'processing_time': processing_time
                }
                
                # Save results after each channel (single CSV that gets updated)
                results_path = os.path.join(output_dir, "memory_bank_results.csv")
                results_df.to_csv(results_path, index=False)
                print(f"Results updated (processing time: {processing_time:.2f}s)")
        except Exception as e:
            print(f"Error processing channel {stream}: {e}")
            # Log the error but continue with other channels
            error_log_path = os.path.join(output_dir, "error_log.txt")
            with open(error_log_path, "a") as error_file:
                error_file.write(f"Error processing {stream}: {str(e)}\n")
        finally:
            # Force cleanup after each channel
            tf.keras.backend.clear_session()
            gc.collect()
    # Calculate total processing time
    total_time = time.time() - overall_start_time
    
    # Calculate and print summary statistics
    if len(results_df) > 0:
        print("\n" + "="*80)
        print(f"SUMMARY OF MEMORY BANK INTEGRATION RESULTS (Total time: {total_time/60:.2f} minutes)")
        print("="*80)
        
        # Average metrics
        avg_f1_with = results_df['f1_with_memory'].mean()
        avg_f1_without = results_df['f1_without_memory'].mean()
        avg_improvement = results_df['improvement'].mean()
        
        print(f"Average F1 Score with memory:    {avg_f1_with:.4f}")
        print(f"Average F1 Score without memory: {avg_f1_without:.4f}")
        print(f"Average improvement:             {avg_improvement:.2f}%")
        
        # Count channels with positive improvement
        improved_channels = results_df[results_df['improvement'] > 0]
        unchanged_channels = results_df[results_df['improvement'] == 0]
        degraded_channels = results_df[results_df['improvement'] < 0]
        
        print(f"\nChannels with improved results:  {len(improved_channels)} ({len(improved_channels)/len(results_df)*100:.1f}%)")
        print(f"Channels with unchanged results: {len(unchanged_channels)} ({len(unchanged_channels)/len(results_df)*100:.1f}%)")
        print(f"Channels with degraded results:  {len(degraded_channels)} ({len(degraded_channels)/len(results_df)*100:.1f}%)")
        
        # Create summary visualization
        create_summary_visualization(results_df, output_dir)
        
        # Also save a summary CSV with just the key metrics
        summary_df = results_df[['channel', 'f1_with_memory', 'f1_without_memory', 'improvement']].copy()
        summary_df = summary_df.sort_values('improvement', ascending=False)
        summary_path = os.path.join(output_dir, "performance_summary.csv")
        summary_df.to_csv(summary_path, index=False)
    
    print(f"\nAll results saved to {output_dir}")
    print("\nMemory bank integration processing complete!")
    
    return output_dir, results_df